In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.ldp_lab;


In [0]:
CREATE VOLUME IF NOT EXISTS workspace.ldp_lab.landing;

In [0]:
%python
import json, random, os
from datetime import datetime, timedelta

CATALOG, SCHEMA = "workspace", "ldp_lab"          # ← adapte si besoin
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
os.makedirs(f"{BASE}/orders", exist_ok=True)
os.makedirs(f"{BASE}/customers_cdc", exist_ok=True)

PAYS     = ["FR", " fr ", "BE", "MA", "ch ", "CA"]      # casse + espaces volontaires
VILLES   = ["Paris","Lyon","Bruxelles","Casablanca","Genève","Montréal","Cergy","Agadir"]
NOMS     = ["Dupont","Martin","Bernard","Petit","Durand","Leroy","Moreau","Simon"]
SEGMENTS = ["particulier","pro","grand_compte"]

def _ecrire(chemin, lignes):
    with open(chemin, "w", encoding="utf-8") as f:
        for l in lignes:
            f.write(json.dumps(l, ensure_ascii=False) + "\n")

def generer_lot(lot: int, n_cmd: int = 200):
    random.seed(1000 + lot)
    t0 = datetime(2026, 7, 1) + timedelta(days=lot)

    # ---- commandes ----
    cmds = []
    for i in range(n_cmd):
        rec = {
            "id_commande":   lot * 10000 + i,
            "id_client":     random.randint(1, 50),
            "pays":          random.choice(PAYS),
            "montant":       round(random.uniform(10, 900), 2),
            "date_commande": (t0 + timedelta(minutes=i * 3)).isoformat(),
            "canal":         random.choice(["web", "mobile", "magasin"]),
        }
        if i % 37 == 0: rec["montant"]     = -rec["montant"]   # ~5 montants négatifs
        if i % 53 == 0: rec["id_commande"] = None              # ~4 id manquants
        cmds.append(rec)
        if i % 71 == 0: cmds.append(dict(rec))                 # ~3 doublons exacts
    _ecrire(f"{BASE}/orders/orders_lot{lot:02d}.json", cmds)

    # ---- CDC clients ----
    if lot == 1:
        evts = [{"op": "c", "id_client": k,
                 "nom": random.choice(NOMS), "ville": random.choice(VILLES),
                 "segment": random.choice(SEGMENTS),
                 "lsn": 100000 + k, "ts": (t0 + timedelta(minutes=k)).isoformat()}
                for k in range(1, 51)]
    else:
        evts = [{"op": random.choices(["u", "c", "d"], weights=[6, 3, 1])[0],
                 "id_client": random.randint(1, 50),
                 "nom": random.choice(NOMS), "ville": random.choice(VILLES),
                 "segment": random.choice(SEGMENTS),
                 "lsn": lot * 100000 + j, "ts": (t0 + timedelta(minutes=j)).isoformat()}
                for j in range(30)]
    _ecrire(f"{BASE}/customers_cdc/cdc_lot{lot:02d}.json", evts)

    print(f"✅ lot {lot} : {len(cmds)} commandes, {len(evts)} événements CDC")

generer_lot(1)

In [0]:
%python
generer_lot(2)

In [0]:
%python
display(dbutils.fs.ls(f"{BASE}/orders"))


In [0]:
%python
display(dbutils.fs.ls(f"{BASE}/customers_cdc"))

In [0]:
select * from read_files("/Volumes/workspace/ldp_lab/landing/orders/", format => "json") limit 10

In [0]:
select *, _metadata.file_name, _metadata.file_modification_time from read_files("/Volumes/workspace/ldp_lab/landing/orders/", format => "json")

In [0]:
select * from workspace.ldp_lab.gold_ca_par_pays_segment

In [0]:
select count(*) from read_files("/Volumes/workspace/ldp_lab/landing/orders/", format => "json") where id_commande is null limit 10

In [0]:
select count(*) from read_files("/Volumes/workspace/ldp_lab/landing/orders/", format => "json") where montant < 0 

In [0]:
select approx_count_distinct(pays) from read_files("/Volumes/workspace/ldp_lab/landing/orders/", format => "json")

In [0]:
select count(distinct pays) from read_files("/Volumes/workspace/ldp_lab/landing/orders/", format => "json")

In [0]:
SELECT count(*)                                   AS total,
       count_if(id_commande IS NULL)              AS id_manquants,
       count_if(montant < 0)                      AS montants_negatifs,
       count(DISTINCT pays)                       AS pays_distincts,
       count(*) - count(DISTINCT id_commande)     AS doublons_approx
FROM read_files('/Volumes/workspace/ldp_lab/landing/orders/', format => 'json');

In [0]:
SELECT pays, count(*) AS n
FROM read_files('/Volumes/workspace/ldp_lab/landing/orders/', format => 'json')
GROUP BY pays ORDER BY n DESC;

In [0]:
DESCRIBE QUERY
SELECT * FROM read_files('/Volumes/workspace/ldp_lab/landing/orders/', format => 'json');

In [0]:
select * from read_files("/Volumes/workspace/ldp_lab/landing/customers_cdc/")

In [0]:
create or refresh streaming table bronze_orders
as select *, _metadata.file_name as filename, current_timestamp() as ingestion_ts from stream read_files("/Volumes/workspace/ldp_lab/landing/orders/", format => "json")

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_orders
(
    CONSTRAINT valid_id_commande EXPECT (id_commande IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_montant EXPECT (montant > 0),
    CONSTRAINT valid_date_commande EXPECT (date_commande is not null) on violation fail update
)
COMMENT "Commandes Silver — avec nettoyage des données"
AS SELECT
     cast(id_commande as bigint) as id_commande,
     cast(date_commande as timestamp) as date_commande,
     upper(trim(pays)) as pays,
     cast(montant as decimal(12,2)) as montant,
     id_client,
     canal,
     current_timestamp as _ingested_at
   FROM STREAM bronze_orders;

CREATE OR REFRESH STREAMING TABLE silver_orders_rejets
(
    CONSTRAINT valid_id_commande EXPECT (id_commande IS NULL) ON VIOLATION DROP ROW
)
COMMENT "Commandes Silver — commandes rejetés"
AS SELECT
     *,
     current_timestamp as _ingested_at
   FROM STREAM bronze_orders;

In [0]:
CREATE OR REFRESH STREAMING TABLE silver_customers
COMMENT "Clients — état courant (SCD 1)";

CREATE FLOW customers_cdc AS AUTO CDC INTO silver_customers
FROM STREAM bronze_customers_cdc
KEYS (id_client)
APPLY AS DELETE WHEN op = 'd'
SEQUENCE BY lsn
COLUMNS * EXCEPT (op, lsn, ts, _source_file, _file_modified_at, _ingested_at)
STORED AS SCD TYPE 1;


CREATE OR REFRESH STREAMING TABLE silver_customers_history
COMMENT "Clients — historique des versions (SCD 2)";

CREATE FLOW customers_cdc_history AS AUTO CDC INTO silver_customers_history
FROM STREAM bronze_customers_cdc
KEYS (id_client)
APPLY AS DELETE WHEN op = 'd'
SEQUENCE BY lsn
COLUMNS * EXCEPT (op, lsn, ts, _source_file, _file_modified_at, _ingested_at)
STORED AS SCD TYPE 2
TRACK HISTORY ON (segment, ville);